In [13]:
import pandas as pd
from typing import List, Union
import itertools

import numpy as np
from collections import defaultdict
import os
import pandas as pd

In [14]:
def estimate_pass_at_k(
    num_samples: Union[int, List[int], np.ndarray],
    num_correct: Union[List[int], np.ndarray],
    k: int
) -> np.ndarray:
    """
    Estimates pass@k of each problem and returns them in an array.
    """

    def estimator(n: int, c: int, k: int) -> float:
        """
        Calculates 1 - comb(n - c, k) / comb(n, k).
        """
        if n - c < k:
            return 1.0
        return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

    if isinstance(num_samples, int):
        num_samples_it = itertools.repeat(num_samples, len(num_correct))
    else:
        assert len(num_samples) == len(num_correct)
        num_samples_it = iter(num_samples)

    return np.array([estimator(int(n), int(c), k) for n, c in zip(num_samples_it, num_correct)])

In [15]:
final_results = []

In [16]:
# Get list of all files in the directory
files = os.listdir('./TestResults/')
json_files = [file for file in files if file.endswith('.jsonl') and 'multi-' in file]
print(len(json_files), "json files found")

24 json files found


In [ ]:
import json

final_results = []
for file_name in json_files:
    model_name =file_name.split('_')[1]
    temp = file_name.split('_')[2].replace('.jsonl','')
    print(model_name, temp)
    with open('./TestResults/' + file_name) as f:
        data = [json.loads(line) for line in f]
    
    results = {}
    for item in data:
        id = item['id']
        language = item['language']
        if language not in results:
            results[language] = defaultdict(list)
        for choice in item['output']:
            results[language][id].append([choice['test_success']=='success',choice['test_vulnerability']=='failure'])
    for language in results.keys():
        current_results = results[language]
        total, correct = [], []
        for result in current_results.values():
            passed = [r[0] for r in result]
            total.append(len(passed))
            correct.append(sum(passed))
        total = np.array(total)
        correct = np.array(correct)

    
        ks = [1,3,5]
        pass_at_k = [(estimate_pass_at_k(total, correct, k).mean())*100
                                for k in ks if (total >= k).all()]
        print(pass_at_k)
            

        total, correct = [], []
        for result in current_results.values():
            passed = [r[1] for r in result]
            total.append(len(passed))
            correct.append(sum(passed))
        total = np.array(total)
        correct = np.array(correct)
            # print(total, correct)

        
        ks = [1,3,5]
        vul_at_k = [(estimate_pass_at_k(total, correct, k).mean())*100
                                for k in ks if (total >= k).all()]
        print(vul_at_k)


        new_security_at_k =[]
        for k in ks:
            total_passed = 0
            for result in current_results.values():
                count = 0
                for i in range(k):
                    if result[i][1] == 1:
                        count += 1
                if count == k:
                    total_passed += 1
            new_security_at_k.append(total_passed/len(current_results.values())*100)
        
        print(new_security_at_k)
        final_results.append([model_name, temp,language,pass_at_k[0], pass_at_k[1], pass_at_k[2], vul_at_k[0], vul_at_k[1], vul_at_k[2], new_security_at_k[0], new_security_at_k[1], new_security_at_k[2]])


   

gpt-4o-mini 0.0
[85.91549295774648, 85.91549295774648, 85.91549295774648]
[80.28169014084507, 80.28169014084507, 80.28169014084507]
[80.28169014084507, 80.28169014084507, 80.28169014084507]
[84.0909090909091, 84.0909090909091, 84.0909090909091]
[78.4090909090909, 78.4090909090909, 78.4090909090909]
[78.4090909090909, 78.4090909090909, 78.4090909090909]
[84.6590909090909, 85.13257575757575, 85.22276334776335]
[78.97727272727273, 79.45075757575756, 79.54094516594516]
[79.54545454545455, 79.54545454545455, 78.4090909090909]
[84.78260869565217, 84.78260869565217, 84.78260869565217]
[79.34782608695652, 79.34782608695652, 79.34782608695652]
[79.34782608695652, 79.34782608695652, 79.34782608695652]
[84.66666666666667, 85.22222222222223, 85.32804232804233]
[78.0, 78.55555555555556, 78.66137566137567]
[78.66666666666666, 77.33333333333333, 77.33333333333333]
[86.55555555555556, 86.66666666666667, 86.66666666666667]
[80.88888888888889, 81.11111111111111, 81.11111111111111]
[81.11111111111111, 81

In [18]:
df = pd.DataFrame(final_results, columns=['Model', 'Test', 'Language', 'Pass@1', 'Pass@3', 'Pass@5', 'Vul@1', 'Vul@3', 'Vul@5', 'New_Security@1', 'New_Security@3', 'New_Security@5'])
df.to_csv('Tests_Results_Multi.csv', index=False)